In [ ]:
import sys, os, pickle
sys.path.append('./src/')

from VAE_variants import VAE, CVAE, CSVAENA, CSVAE, HCSVAENA, HCSVAE, DLVAE, SDIVA, CCVAE
from VAE_trainers import EpochPyroTrainer, AdversarialEpochPyroTrainer, ThresholdPyroTrainer, AdversarialThresholdPyroTrainer
from matplotlib.colors import LinearSegmentedColormap
from sklearn.naive_bayes import GaussianNB
from tqdm import tqdm, trange
import pyro.distributions as dist

import torch, pyro
import numpy as np
import matplotlib.pyplot as plt
import pyro.optim as opt
import seaborn as sns

demo_epochs=5

np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

cmap = LinearSegmentedColormap.from_list("cmap", ["#F23E2E", "#5888A6"])

## Data

In [ ]:
import torch.utils.data as utils

#################### DATASET PARAMS #########################################################################################
n_samples = 30000
batch_size = 64
##################################################################################################################################

# Generate latent variables
z_true = np.random.randn(n_samples)       
w_true = np.random.randn(n_samples)       

# Observation: x = z + w, so x ~ N(0,2)
x = z_true + w_true

# Binary label: y = 1 if w > 0, else 0
y = (w_true > 0).astype(np.int64)
y = torch.tensor(y).float().reshape(-1,1)

# Convert to PyTorch tensors
x = torch.tensor(x, dtype=torch.float32).unsqueeze(1)  # shape: [N,1]
y = y  # shape: [N,1]


dataset = utils.TensorDataset(x, y, torch.hstack((torch.FloatTensor(z_true.reshape(-1,1)), torch.FloatTensor(w_true.reshape(-1,1)))))
train_set, test_set = utils.random_split(dataset, [0.5, 0.5])  
train_set, test_set = utils.TensorDataset(*train_set[:]), utils.TensorDataset(*test_set[:])
train_loader, test_loader = torch.utils.data.DataLoader(train_set, shuffle=True, batch_size=batch_size), torch.utils.data.DataLoader(test_set, shuffle=False, batch_size=batch_size)

In [ ]:
plt.hist(w_true, color='#9E666F', alpha=0.6, bins=75)
plt.xlim(-6,6)
plt.show()

In [ ]:
plt.hist(z_true, color='#9E666F', alpha=0.6, bins=75)
plt.xlim(-6,6)
plt.show()

In [ ]:
plt.hist(x.flatten().numpy(), color='#9E666F', alpha=0.6, bins=75)
plt.xlim(-6,6)
plt.show()

In [ ]:
orig_data_score = GaussianNB().fit(train_set[:][0], train_set[:][1].squeeze(-1)).score(train_set[:][0], train_set[:][1].squeeze(-1))
print(f'Bayes classifier acc - Combined: {np.round(orig_data_score, 2)}')

## CSVAE - No Adv.

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvaena = CSVAENA(1, [1], latent_dim=1, w_dim=1, hidden_dim=8)
csvaena_trainer = EpochPyroTrainer(demo_epochs, csvaena, train_loader, test_loader)
csvaena_trainer.train()

In [ ]:
csvaena_trainer._predictive_setup(s=1)
preds = csvaena_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
print(dist.Normal(recons.mean(), recons.std()).log_prob(test_set[:][0]).mean())

In [ ]:
sns.stripplot(recons.flatten().numpy(), orient='y', c=test_set[:][1].numpy(), cmap=cmap, size=5)
plt.xlim(-6,6)
plt.show()

In [ ]:
sns.stripplot(z_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap, alpha=0.6)
plt.xlim(-6,6)
plt.show()

In [ ]:
sns.stripplot(w_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap, alpha=0.6)
plt.xlim(-6,6)
plt.show()

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == test_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 2)}')
print(f'Dfif: {np.round(abs(orig_data_score - score), 2)}' )

In [ ]:
plt.hist(w_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-1,6)
plt.axis('off')

In [ ]:
plt.hist(z_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-6,6)
plt.axis('off')

In [ ]:
# Plot X Samples
plt.hist(recons.flatten().numpy(), color='purple', alpha=0.6, bins=500)
plt.title("Reconstructed X Samples")
plt.show()

## CSVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvae = CSVAE(1, [1], latent_dim=1, w_dim=1, hidden_dim=8, recon_weight=2.5, adversarial_weight=20, w_kl_weight=5e-1)
csvae_trainer = AdversarialEpochPyroTrainer(demo_epochs, 1, 1, csvae, train_loader, test_loader)
csvae_trainer.train()

In [ ]:
csvae_trainer._predictive_setup(s=1)
preds = csvae_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
print(dist.Normal(recons.mean(), recons.std()).log_prob(test_set[:][0]).mean())

In [ ]:
# Plot X Samples
sns.stripplot(recons.flatten().numpy(), orient='y', c=test_set[:][1].numpy(), cmap=cmap)
plt.xlim(-6,6)
plt.show()

In [ ]:
sns.stripplot(z_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap)
plt.xlim(-6,6)
plt.show()

In [ ]:
sns.stripplot(w_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap)
plt.xlim(-6,6)
plt.show()

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == train_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 2)}')
print(f'Dfif: {np.round(abs(orig_data_score - score), 2)}' )

In [ ]:
plt.hist(z_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-6,6)
plt.axis('off')

In [ ]:
plt.hist(w_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-1,6)
plt.axis('off')

In [ ]:
# Plot X Samples
plt.hist(recons.flatten().numpy(), color='purple', alpha=0.6, bins=500)
plt.title("Reconstructed X Samples")
plt.show()

## HCSVAE - No Adv.

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

hcsvaena = HCSVAENA(1, [1], latent_dim=1, w_dim=1, hidden_dim=8, w_kl_weight=5e-1)
hcsvaena_trainer = EpochPyroTrainer(demo_epochs, hcsvaena, train_loader, test_loader)
hcsvaena_trainer.train()

In [ ]:
hcsvaena_trainer._predictive_setup(s=1)
preds = hcsvaena_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
print(dist.Normal(recons.mean(), recons.std()).log_prob(test_set[:][0]).mean())

In [ ]:
# Plot X Samples
sns.stripplot(recons.flatten().numpy(), orient='y', c=test_set[:][1].numpy(), cmap=cmap, size=5)
plt.xlim(-6,6)

plt.show()

In [ ]:
sns.stripplot(z_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap, alpha=0.6)
plt.xlim(-6,6)

plt.show()

In [ ]:
sns.stripplot(w_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap, alpha=0.6)
plt.xlim(-1,6)

plt.show()

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == test_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Dfif: {np.round(abs(np.round(orig_data_score, 2) - np.round(score, 4)), 4)}' )

In [ ]:
plt.hist(w_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-1,6)
plt.axis('off')

In [ ]:
plt.hist(z_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-6,6)
plt.axis('off')

In [ ]:
# Plot X Samples
plt.hist(recons.flatten().numpy(), color='purple', alpha=0.6, bins=500)
plt.title("Reconstructed X Samples")
plt.show()

## HCSVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

hcsvae = HCSVAE(1, [1], latent_dim=1, w_dim=1, hidden_dim=8, recon_weight=2.5, adversarial_weight=20, w_kl_weight=5e-1)
hcsvae_trainer = AdversarialEpochPyroTrainer(demo_epochs, 1, 1, hcsvae, train_loader, test_loader)
hcsvae_trainer.train()

In [ ]:
hcsvae_trainer._predictive_setup(s=1)
preds = hcsvae_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
print(dist.Normal(recons.mean(), recons.std()).log_prob(test_set[:][0]).mean())

In [ ]:
# Plot X Samples
sns.stripplot(recons.flatten().numpy(), orient='y', c=test_set[:][1].numpy(), cmap=cmap, size=5)
plt.axis('off')
plt.xlim(-6,6)

plt.show()

In [ ]:
sns.stripplot(z_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.xlim(-6,6)

plt.show()

In [ ]:
sns.stripplot(w_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.xlim(-6,6)

plt.show()

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == test_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Dfif: {np.round(abs(np.round(orig_data_score, 2) - np.round(score, 4)), 4)}' )

In [ ]:
plt.hist(z_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-6,6)
plt.axis('off')


In [ ]:
plt.hist(w_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-1,6)
plt.axis('off')


In [ ]:
# Plot X Samples
plt.hist(recons.flatten().numpy(), color='purple', alpha=0.6, bins=500)
plt.title("Reconstructed X Samples")
plt.show()

## DIVA

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

diva = SDIVA(1, [1], latent_dim=1, w_dim=1, hidden_dim=8)
diva_trainer = EpochPyroTrainer(demo_epochs, diva, train_loader, test_loader)
diva_trainer.train()

In [ ]:
diva_trainer._predictive_setup(s=1)
preds = diva_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
print(dist.Normal(recons.mean(), recons.std()).log_prob(test_set[:][0]).mean())

In [ ]:
# Plot X Samples
sns.stripplot(recons.flatten().numpy(), orient='y', c=test_set[:][1].numpy(), cmap=cmap, size=5)
plt.axis('off')
plt.xlim(-6,6)

plt.show()

In [ ]:
sns.stripplot(z_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.xlim(-6,6)

plt.show()

In [ ]:
sns.stripplot(w_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.xlim(-6,6)

plt.show()

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == test_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Dfif: {np.round(abs(orig_data_score - score), 4)}' )

In [ ]:
plt.hist(w_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-1,6)
plt.axis('off')

In [ ]:
plt.hist(z_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-6,6)
plt.axis('off')

In [ ]:
# Plot X Samples
plt.hist(recons.flatten().numpy(), color='purple', alpha=0.6, bins=500)
plt.title("Reconstructed X Samples")
plt.show()

## CCVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

ccvae = CCVAE(1, [1], latent_dim=1, w_dim=1, hidden_dim=8)
ccvae_trainer = EpochPyroTrainer(demo_epochs, ccvae, train_loader, test_loader)
ccvae_trainer.train()

In [ ]:
ccvae_trainer._predictive_setup(s=1)
preds = ccvae_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
print(dist.Normal(recons.mean(), recons.std()).log_prob(test_set[:][0]).mean())

In [ ]:
# Plot X Samples
sns.stripplot(recons.flatten().numpy(), orient='y', c=test_set[:][1].numpy(), cmap=cmap, size=5)
plt.axis('off')
plt.xlim(-6,6)

plt.show()

In [ ]:
sns.stripplot(z_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.xlim(-6,6)

plt.show()

In [ ]:
sns.stripplot(w_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.xlim(-6,6)

plt.show()

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == test_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Dfif: {np.round(abs(orig_data_score - score), 4)}' )

In [ ]:
plt.hist(w_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-1,6)
plt.axis('off')

In [ ]:
plt.hist(z_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-6,6)
plt.axis('off')

In [ ]:
# Plot X Samples
plt.hist(recons.flatten().numpy(), color='purple', alpha=0.6, bins=500)
plt.title("Reconstructed X Samples")
plt.show()

## DISCoVeR

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

dlvae = DLVAE(1, [1], latent_dim=1, w_dim=1, hidden_dim=8, recon_weight=7e-1, recon_weight_z=3e-1, w_kl_weight=2e-1, z_kl_weight=7e-1, adversarial_weight=8e-1)
dlvae_trainer = AdversarialEpochPyroTrainer(demo_epochs, 1, 1, dlvae, train_loader, test_loader)
dlvae_trainer.train()

In [ ]:
dlvae_trainer._predictive_setup(s=1)
preds = dlvae_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons_z = preds['rec_z'][0, 0].cpu()
recons_w = preds['rec_w'][0, 0].cpu()

In [ ]:
print(dist.Normal(recons_w.mean(), recons_w.std()).log_prob(test_set[:][0]).mean())
print(dist.Normal(recons_z.mean(), recons_z.std()).log_prob(test_set[:][0]).mean())

In [ ]:
# Plot X Samples - From Z
sns.stripplot(recons_w.flatten().numpy(), orient='y', c=test_set[:][1].numpy(), cmap=cmap, size=5)
plt.axis('off')
plt.xlim(-6,6)

plt.show()

In [ ]:
# Plot X Samples - From Z
sns.stripplot(recons_z.flatten().numpy(), orient='y', c=test_set[:][1].numpy(), cmap=cmap, size=5)
plt.axis('off')
plt.xlim(-6,6)


plt.show()

In [ ]:
sns.stripplot(z_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.xlim(-6,6)

plt.show()

In [ ]:
sns.stripplot(w_s, orient='y', c=test_set[:][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.xlim(-6,6)

plt.show()

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == test_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Dfif: {np.round(abs(np.round(orig_data_score, 2) - np.round(score, 4)), 4)}' )

In [ ]:
plt.hist(z_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-6,6)
plt.axis('off')

In [ ]:
plt.hist(w_s.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-6,6)
plt.axis('off')

In [ ]:
plt.hist(recons_z.flatten().numpy(), color='#9E666F', alpha=0.6, bins=500)
plt.xlim(-6,6)
plt.axis('off')

In [ ]:
# Plot X Samples
fig, ax = plt.subplots(1,2,figsize=(16,12), sharex=True, sharey=True)

ax[0].hist(recons_w.flatten().numpy(), color='purple', alpha=0.6, bins=500)
ax[0].set_title("Reconstructed X Samples")

ax[1].hist(recons_z.flatten().numpy(), color='purple', alpha=0.6, bins=500)
ax[1].set_title("Reconstructed X Samples - From Z")

plt.show()

In [ ]:
df_lats = []

In [ ]:
# Expand for conditionals
idxs = [555, 768, 433, 93]
dlvae_trainer._predictive_setup(s=1000)
preds = dlvae_trainer.predictive(*dlvae_trainer._send_args_to_device(dlvae_trainer.test_loader.dataset[:1000], dlvae_trainer.device))
z_s = preds['z'].cpu()
w_s = preds['w'].cpu()

for idx in idxs:
    x, y = test_set[idx][:2]
    x, y = x.item(), y.item()

    z = z_s[:, idx, :]
    w = w_s[:, idx, :]

    for i in range(1000):
        df_lats.append([x, z[i].item(), w[i].item(), 'DISCoVeR'])

In [ ]:
from scipy.stats import norm, truncnorm
import pandas as pd

# True posterior
for idx in idxs:
    x, y = test_set[idx][:2]
    x, y = x.item(), y.item()

    if y == 1:
        a, b = (0 - x/2) / np.sqrt(0.5), np.inf

    else:
        a, b = -np.inf, (0 - x/2) / np.sqrt(0.5)
    
    z = norm.rvs(loc=x/2, scale=np.sqrt(0.5), size=1000)
    w = truncnorm.rvs(loc=x/2, scale=np.sqrt(0.5), a=a, b=b, size=1000)

    for i in range(1000):
        df_lats.append([x, z[i], w[i], 'True Posterior'])

In [ ]:
import pandas as pd

df = pd.DataFrame(df_lats, columns=['X', 'Z', 'W', 'Model'])
linestyles = ['solid', 'solid']
alphas = [1,1] 

palette = sns.color_palette("colorblind", n_colors=8)
palette[2], palette[-2] = palette[-2], palette[2]
palette[-1] = (0,0,0)
palette = palette[-2:]


from pyfonts import load_font

# load font
font = load_font(
   font_url="https://github.com/stevenpetryk/computer-modern/blob/main/src/cmunrm.ttf?raw=true"
)

# True posterior: Black, straight
# different linepatterns / lower alpha for other methods
# pick vibrant color for us

fig, ax = plt.subplots(2,4,figsize=(10,5), sharey=False, sharex=True)

x = df['X'].unique()[3]
y = 1

for x, y, axtrack, count in zip(list(df['X'].unique()), [0,0,1,1], [(0,1), (2,3), (0,1), (2,3)], range(4)):

    cur_ax_1, cur_ax_2 = ax[count // 2][axtrack[0]], ax[count // 2][axtrack[1]]
    
    
    xs = np.linspace(-5,5,1000)
    
    for i, model in enumerate(df['Model'].unique()):
    
        if model != "True Posterior":
            df_sub = df[(df['X'] == x) & (df['Model'] == model)]
            mu_z, sigma_z = df_sub['Z'].mean(), df_sub['Z'].std() 
            mu_w, sigma_w = df_sub['W'].mean(), df_sub['W'].std()
    
        else:
            mu_z, sigma_z = x/2, np.sqrt(1/2) 
            mu_w, sigma_w = x/2, np.sqrt(1/2) 
    
        
    
        if y == 1:
            a, b = (0 - x/2) / sigma_w, np.inf
    
        else:
            a, b = -np.inf, (0 - x/2) / sigma_w
    
        
        p_z = norm.pdf(xs, mu_z, sigma_z)
        if model == 'True Posterior':
            p_w = truncnorm.pdf(xs, loc=mu_w, scale=sigma_w, a=a, b=b)
    
        else:
            p_w = norm.pdf(xs, mu_w, sigma_w)
    
        cur_ax_1.plot(xs, p_z, c=palette[i], label=f'{model}', linestyle=linestyles[i], alpha=alphas[i])
        cur_ax_2.plot(xs, p_w, c=palette[i], linestyle=linestyles[i], alpha=alphas[i])
    
    
    
    cur_ax_1.set_ylim((0,1.5))
    cur_ax_2.set_ylim((0,1.5))
    
    cur_ax_1.spines[['top', 'right']].set_visible(False)
    cur_ax_2.spines[['top', 'right']].set_visible(False)
    
    cur_ax_2.set_yticks([])
    
    
    cur_ax_1.set_xticklabels(cur_ax_1.get_xticklabels(), font=font, fontsize=6)
    cur_ax_2.set_xticklabels(cur_ax_2.get_xticklabels(), font=font, fontsize=6)
    
    cur_ax_1.set_yticklabels(cur_ax_1.get_yticklabels(), font=font, fontsize=6)
    cur_ax_2.set_yticklabels(cur_ax_2.get_yticklabels(), font=font, fontsize=6)
    
    cur_ax_1.set_title('Z', font=font, fontsize=6)
    cur_ax_2.set_title('W', font=font, fontsize=6)
    
    
    cur_ax_1.set_title(f'Z | X = {np.round(x, 3)}', font=font, fontsize=6)
    cur_ax_2.set_title(f'W | X = {np.round(x, 3)}, Y = {int(y)}', font=font, fontsize=6)

    cur_ax_1.legend()

plt.show()